In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

boto3==1.24.59
pandas==1.2.4
catboost==1.0.4
scikit_learn==0.24.1

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import catboost as cb
import sklearn.metrics as skm
import boto3
import pickle

# get metric
def get_metric_by_year_month(target, y_hat, str_eval_metric):
    if str_eval_metric == 'AUC':
        return skm.roc_auc_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'PRAUC':
        return skm.average_precision_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'Logloss':
        return skm.log_loss(y_true=target, y_pred=y_hat)
    elif str_eval_metric == 'F1':
        return skm.f1_score(y_true=target, y_pred=y_hat)
    else:
        pass

# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# constants
str_project = '20231010-gen-xii'
str_dirname_output = './output'
    
# create output dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

# get index of array
int_idx_array = int(os.environ['AWS_BATCH_JOB_ARRAY_INDEX'])
print(f'Array index: {int_idx_array}')

# get the target
str_filename = 'df_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
list_targets = list(pd.read_csv(str_uri)['Target'])
str_target = list_targets[int_idx_array]
print(f'Target: {str_target}')

# get the target
str_filename = 'df_monitoring_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
list_cols = [
    'UniqueID',
    str_target,
]
df_target = pd.read_csv(str_uri, usecols=list_cols)
# rename
dict_rename = {
    'UniqueID': 'uniqueid',
}
df_target.rename(columns=dict_rename, inplace=True)
# set to int
df_target['uniqueid'] = df_target['uniqueid'].astype(int)
# rm dups
df_target.drop_duplicates(subset=['uniqueid'], keep='last', inplace=True)
print(df_target.shape[0])

###############################################################################
# HYPERPARAMETERS
###############################################################################
# get df_hyperparameters
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
df = pd.read_csv(str_uri)
# convert to dict
dict_hyperparameters = dict(zip(df['keys'], df['values']))

# get number of iterations
int_n_iterations = int(dict_hyperparameters['INT_N_ITERATIONS'])
print(f'Iterations: {int_n_iterations}')

# get filename for training
str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
print(f'Training filename: {str_filename_train}')

# get filename for valid
str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
print(f'Valid filename: {str_filename_valid}')

# get proportion of iterations to use as early stopping rounds
flt_prop_early_stopping = float(dict_hyperparameters['PROP_EARLY_STOPPING'])
print(f'Proportion early stopping: {flt_prop_early_stopping}')

# get eval metric
str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
print(f'Eval metric: {str_eval_metric}')

# get the number of tuning jobs
int_n_tuning_jobs = int(dict_hyperparameters['INT_N_TUNING_JOBS'])
print(f'N Tuning Jobs: {int_n_tuning_jobs}')

# get the model variant
str_variant = dict_hyperparameters['STR_VARIANT']
print(f'Variant: {str_variant}')

##################################################################################

# get list of features
print('Importing list_of_starting_features...')
str_filename = 'df_cols_in_model.csv'
str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
list_cols_model = list(pd.read_csv(str_uri)['feature'])
# add unique id for joining
list_cols_import = ['uniqueid'] + list_cols_model

# read training data
print('Reading training data...')
str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_variant}/00_preprocessing/02_make_dfs/{str_filename_train}'
df = pd.read_parquet(str_uri, columns=list_cols_import)
# set int
df['uniqueid'] = df['uniqueid'].astype(int)
# rm dups
df.drop_duplicates(subset=['uniqueid'], keep='last', inplace=True)
print(df.shape[0])

# join to get new target
print('Joining training data to early indicator target to fit model...')
df = pd.merge(
    left=df,
    right=df_target,
    on='uniqueid',
    how='inner',
)
print(df.shape[0])

# get the non numeric feats
print('Getting list of non-numeric columns...')
list_cols_non_numeric = []
for col in list_cols_model:
    if df[col].dtype not in ['int64','float64']:
        list_cols_non_numeric.append(col)

# build model
print('Building model...')
# pool data
pool_train = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# read validation data
print('Reading validation data...')
str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_variant}/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri, columns=list_cols_import)
# set int
df['uniqueid'] = df['uniqueid'].astype(int)
# rm dups
df.drop_duplicates(subset=['uniqueid'], keep='last', inplace=True)
print(df.shape[0])

# join to get new target
print('Joining validation data to early indicator target to fit model...')
df = pd.merge(
    left=df,
    right=df_target,
    on='uniqueid',
    how='inner',
)
print(df.shape[0])

# pool data
pool_valid = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# constraints
dict_monotone_constraints = {
    # better
    'fltgrossmonthly__income_sum': -1, # as income increases, prediction gets better
    'fltapproveddowntotal__app': -1,
    'fltdowncash__app': -1,
    'bookvalue__app': -1,
    'ENG-dealership_age': -1,
    'bookvalue__app': -1,
    'fltdowncash__app': -1,
    'fltapproveddowntotal__app': -1,
    # worse
    'fltgrossmonthly__income_count': 1, # as count of income increases, prediction gets worse
    'ENG-loan_to_value': 1,
    'ENG-payment_to_income': 1,
    'ENG-vehicle_age': 1,
    'fltadvance__app': 1,
    'bigmileage_odometer__app': 1,
    'amtfinanced__app': 1,
    'miles_odometer__app': 1,
    'pti__app': 1,
    'fltadvance__app': 1,
    'ENG-loan_to_value': 1,
    'amtfinanced__app': 1,
}
# ensure features are in list_cols_model
dict_monotone_constraints = {key: val for key, val in dict_monotone_constraints.items() if key in list_cols_model}

# get lr
print('Getting learning rates...')
list_flt_learning_rate = list(np.around(np.linspace(0.001, 0.999, int_n_tuning_jobs), 4))

# iterate through LRs
for a, flt_learning_rate in enumerate(list_flt_learning_rate):
    # init class
    cls_model_inference = cb.CatBoostClassifier(
        task_type='CPU',
        nan_mode='Min',
        random_state=42,
        eval_metric=str_eval_metric,
        iterations=int_n_iterations,
        learning_rate=flt_learning_rate,
        class_weights=None,
        monotone_constraints=dict_monotone_constraints,
    )
    # fit
    cls_model_inference.fit(
        pool_train,
        eval_set=[pool_valid],
        verbose=100,
        use_best_model=True,
        early_stopping_rounds=int(round(int_n_iterations*flt_prop_early_stopping)), 
    )
    ################################################################################################
    # GET TRAINING EVAL METRIC
    ################################################################################################
    print('Getting training eval metric...')
    # import data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_variant}/00_preprocessing/02_make_dfs/{str_filename_train}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)
    # set int
    df['uniqueid'] = df['uniqueid'].astype(int)
    # rm dups
    df.drop_duplicates(subset=['uniqueid'], keep='last', inplace=True)
    print(df.shape[0])
    
    # join to get new target
    print('Joining training data to early indicator target to get training eval metric...')
    df = pd.merge(
        left=df,
        right=df_target,
        on='uniqueid',
        how='inner',
    )

    # get predictions
    if str_eval_metric in ['AUC','PRAUC','Logloss']:
        # probabilities
        df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
    elif str_eval_metric in ['F1']:
        # class
        df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
    else:
        pass

    # save train predictions
    y_hat_train = list(df['y_hat'])

    # get eval metric - train
    if str_eval_metric == 'AUC':
        flt_eval_metric_train = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'PRAUC':
        flt_eval_metric_train = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'Logloss':
        flt_eval_metric_train = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
    elif str_eval_metric == 'F1':
        flt_eval_metric_train = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

    # save memory
    del df

    ################################################################################################
    # GET VALIDATION EVAL METRIC (WEIGHTED)
    ################################################################################################
    print('Getting validation eval metric...')

    # import data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_variant}/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)
    # set int
    df['uniqueid'] = df['uniqueid'].astype(int)
    # rm dups
    df.drop_duplicates(subset=['uniqueid'], keep='last', inplace=True)
    print(df.shape[0])
    
    # join to get new target
    print('Joining validation data to early indicator target to get eval metric...')
    df = pd.merge(
        left=df,
        right=df_target,
        on='uniqueid',
        how='inner',
    )
    print(df.shape[0])

    # get predictions
    if str_eval_metric in ['AUC','PRAUC','Logloss']:
        # probabilities
        df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
    elif str_eval_metric in ['F1']:
        # class
        df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
    else:
        pass

    # save valid predictions
    y_hat_valid = list(df['y_hat'])

    # get eval metric - train
    if str_eval_metric == 'AUC':
        flt_eval_metric_valid = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'PRAUC':
        flt_eval_metric_valid = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'Logloss':
        flt_eval_metric_valid = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
    elif str_eval_metric == 'F1':
        flt_eval_metric_valid = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

    # save memory
    del df

    ################################################################################################
    # PICKLE MODEL AND PREDICTIONS
    ################################################################################################
    print('Pickling model and predictions...')
    flt_diff = abs(flt_eval_metric_train - flt_eval_metric_valid)
    dict_output = {
        'model_inference': cls_model_inference,
        'y_hat_train': y_hat_train,
        'flt_eval_metric_train': flt_eval_metric_train,
        'y_hat_valid': y_hat_valid,
        'flt_eval_metric_valid': flt_eval_metric_valid,
        'diff': flt_diff,
        'best_iteration': cls_model_inference.get_best_iteration(),
    }
    str_filename = f'dict_model_inference_{a}.pkl'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    pickle.dump(dict_output, open(str_local_path, 'wb'))
    # upload
    str_bucket_path = f'09_early_indicators/{str_variant}/models/{str_target}/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

    ################################################################################################
    # SAVE CSV
    ################################################################################################
    # create output data frame
    print('Creating output data frame...')
    dict_row = {
        'iteration': a,
        'learning_rate': flt_learning_rate,
        'flt_eval_metric_train': flt_eval_metric_train,
        'flt_eval_metric_valid': flt_eval_metric_valid,
        'diff': flt_diff,
        'best_iteration': cls_model_inference.get_best_iteration(),
    }
    df = pd.DataFrame(dict_row, index=[0])
    # save
    str_filename = f'df_output_{a}.csv'
    str_uri = f's3://{str_project}/09_early_indicators/{str_variant}/dfs/{str_target}/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-pd-monitoring

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  66.56kB
Step 1/7 : FROM python:3.9
 ---> e301b6ca4781
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 15f5bd9e0853
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 8d54816abe58
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> 3205672eba56
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> 1b353925e949
Step 6/7 : COPY script.py .
 ---> a6013b625dc1
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in 7d8c2b2eea2a
Removing intermediate container 7d8c2b2eea2a
 ---> ed0dcc91f67b
Successfully built ed0dcc91f67b
Successfully tagged genxii-pd-monitoring:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-monitoring' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-monitoring]
92be02c6867c: Preparing
ae727d5d1015: Preparing
10b48375735b: Preparing
444598740dd7: Preparing
0b5252a0a066: Preparing
370e298d868b: Preparing
258eec9449ed: Preparing
8f728ab4090c: Preparing
84f540ade319: Preparing
9fe4e8a1862c: Preparing
909275a3eaaa: Preparing
f3f47b3309ca: Preparing
1a5fc1184c48: Preparing
9fe4e8a1862c: Waiting
370e298d868b: Waiting
258eec9449ed: Waiting
8f728ab4090c: Waiting
909275a3eaaa: Waiting
84f540ade319: Waiting
f3f47b3309ca: Waiting
1a5fc1184c48: Waiting
10b48375735b: Layer already exists
444598740dd7: Layer already exists
ae727d5d1015: Layer already exists
0b5252a0a066: Layer already exists
258eec9449ed: Layer already exists
370e298d868b: Layer already exists
8f728ab4090c: Layer already exists
84f540ade319: Layer already exists
909275a3eaaa: Layer already exists
9fe4e8a1862c: Layer already exists
f3f47b3309ca: Layer already exists
1a5fc1184c48: Layer already e

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass